---
# Consistencia Semántica de Tópicos con Embeddings
## Python + Sentence Embeddings
---

En **5b** le pedimos a un LLM que **leyera y calificara** la coherencia de cada tópico en una escala de 1 a 5. Encontramos una limitación real: el modelo tiende a concentrar casi todos sus puntajes en 5, lo que deja poco margen para distinguir entre modelos con $k=12$, $k=20$ y $k=25$.

En este notebook usamos un enfoque distinto y **puramente matemático**: en vez de pedirle a un LLM que *opine* sobre la coherencia, medimos la **distancia real entre las palabras de un tópico en un espacio vectorial** (*embeddings*).

La idea central:

> Si las palabras de un tópico son **semánticamente cercanas** en el espacio de embeddings (alta similitud coseno promedio entre ellas), el tópico es coherente. Si están dispersas (baja similitud), el tópico mezcla conceptos poco relacionados.

A diferencia de pedirle una nota a un LLM, este método produce un número **continuo** (no un entero de 1 a 5), lo que debería darnos una comparación entre valores de $k$ mucho más informativa.

> 🖥️ **Sobre el hardware:** este notebook es mucho más liviano que 5b. El modelo de embeddings que usamos (~110M parámetros) es unas 30 veces más pequeño que el LLM de 5b, y calcular una similitud coseno es prácticamente instantáneo. Corre en pocos segundos incluso en **CPU** — no necesitas GPU, aunque igual la usaremos automáticamente si está disponible.

## 🎯 Objetivos de aprendizaje

- Entender la diferencia entre pedirle a un LLM que **evalúe** coherencia (5b) y **medirla matemáticamente** con embeddings (este notebook).
- Generar embeddings de palabras con un modelo de Hugging Face (`sentence-transformers`).
- Calcular la similitud coseno promedio entre las palabras de un tópico como una métrica continua de coherencia.
- Comparar esta métrica entre $k=12$, $k=20$ y $k=25$, y contrastar los resultados con los de 5b.

## 🧮 ¿Qué modelo de embeddings usamos?

Usaremos **`sentence-transformers/all-mpnet-base-v2`** (MPNet).

Es una alternativa a modelos más grandes como `BAAI/bge-large-en-v1.5` (BGE), con ventajas prácticas para este notebook:

- **Liviano (~420 MB, 110M parámetros)** frente a BGE-large (~1.3 GB, 335M parámetros) — se descarga y carga notablemente más rápido, igual que preferimos un modelo más chico en 5b.
- **No requiere *prompts* especiales.** BGE recomienda anteponer frases específicas como `"Represent this sentence for searching relevant passages: "` para obtener su mejor rendimiento en tareas de *búsqueda*; nuestra tarea (medir qué tan cerca están las palabras de un mismo tópico) no es una tarea de búsqueda, y MPNet está entrenado de forma general para similitud semántica sin necesitar ese tipo de ajuste.
- **Ampliamente probado**: es uno de los modelos de `sentence-transformers` más usados y no tiene restricciones de acceso.

BGE suele obtener puntajes ligeramente más altos en benchmarks de *retrieval* (búsqueda de documentos), pero para medir similitud semántica general entre palabras — que es justamente lo que necesitamos — la diferencia práctica es mínima.

> 📌 El nombre del modelo está aislado en una sola variable (`MODEL_NAME`). Si quieres comparar con BGE, cambia `MODEL_NAME` a `"BAAI/bge-large-en-v1.5"` — el resto del notebook funciona igual.

## 📦 Instalación de las librerías necesarias

In [ ]:
!pip install -q sentence-transformers pandas numpy matplotlib scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

## 🧠 Cargar el modelo de embeddings

Igual que en 5b, detectamos automáticamente el hardware disponible. A diferencia del LLM de 5b, este modelo es lo suficientemente liviano como para que la diferencia entre GPU y CPU sea de segundos, no de minutos.

In [ ]:
MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Usando dispositivo: {device}")
modelo_embeddings = SentenceTransformer(MODEL_NAME, device=device)

## 📂 Cargar los tópicos exportados desde R (Unidad 5a)

Los tres archivos deben estar en el mismo directorio de trabajo que este notebook.

> 📌 **Si estás en Google Colab:** sube los tres archivos (`lda_topterms_k12.csv`, `lda_topterms_k20.csv`, `lda_topterms_k25.csv`) usando el panel de archivos (ícono de 📁 en la barra lateral izquierda) antes de ejecutar la siguiente celda — ver la nota detallada en 5b si no lo has hecho antes.

In [ ]:
topterms_12 = pd.read_csv("lda_topterms_k12.csv")
topterms_20 = pd.read_csv("lda_topterms_k20.csv")
topterms_25 = pd.read_csv("lda_topterms_k25.csv")

topterms_12.head()

## ✂️ Palabras clave por tópico

La misma función utilizada en 5b: obtiene las $n$ palabras con mayor probabilidad ($\beta$) de un tópico.

In [ ]:
def get_topic_words(df, topic_id, n=15):
    palabras = (
        df[df["topic"] == topic_id]
        .sort_values("beta", ascending=False)
        .head(n)["term"]
        .tolist()
    )
    return palabras

get_topic_words(topterms_20, topic_id=1)

## 📐 De palabras a un puntaje de coherencia

Para un tópico con $n$ palabras, el procedimiento es:

1. Generar un embedding (vector de 768 dimensiones) para cada una de las $n$ palabras.
2. Calcular la **similitud coseno** entre cada par de palabras — esto produce una matriz de $n \times n$ similitudes.
3. Promediar todas las similitudes, **excluyendo la diagonal** (la similitud de una palabra consigo misma siempre es 1, y no aporta información).

El resultado es un número entre -1 y 1 (en la práctica, casi siempre entre 0 y 1 para palabras en inglés): mientras más alto, más cercanas están las palabras entre sí en el espacio semántico, y por lo tanto, más coherente es el tópico.

> 🧠 A diferencia del puntaje 1-5 de un LLM, este es un número **continuo** — dos tópicos pueden tener 0.31 y 0.34 de similitud promedio, una diferencia real que un LLM forzado a elegir un entero entre 1 y 5 probablemente redondearía al mismo puntaje.

In [ ]:
def coherencia_por_embeddings(palabras):
    embeddings = modelo_embeddings.encode(palabras)
    matriz_similitud = cosine_similarity(embeddings)

    # excluimos la diagonal (similitud de cada palabra consigo misma = 1)
    n = len(palabras)
    mascara = ~np.eye(n, dtype=bool)
    similitudes = matriz_similitud[mascara]

    return similitudes.mean()


# ejemplo con un solo tópico
palabras_ejemplo = get_topic_words(topterms_20, topic_id=1)
coherencia_por_embeddings(palabras_ejemplo)

## 🔁 Aplicar a todos los tópicos de un modelo

A diferencia de 5b, aquí no hay llamadas a un LLM ni *prompts* que parsear — solo embeddings y álgebra lineal, por lo que esta celda debería ejecutarse en segundos para los 57 tópicos combinados.

In [ ]:
def evaluate_model_topics_embeddings(df, k, n_words=15):
    resultados = []
    for topic_id in sorted(df["topic"].unique()):
        palabras = get_topic_words(df, topic_id, n=n_words)
        coherencia = coherencia_por_embeddings(palabras)
        resultados.append({
            "k": k,
            "topic": topic_id,
            "coherencia_embeddings": coherencia,
            "palabras": ", ".join(palabras),
        })
    return pd.DataFrame(resultados)

In [ ]:
resultados_12 = evaluate_model_topics_embeddings(topterms_12, k=12)
resultados_20 = evaluate_model_topics_embeddings(topterms_20, k=20)
resultados_25 = evaluate_model_topics_embeddings(topterms_25, k=25)

resultados_todos = pd.concat([resultados_12, resultados_20, resultados_25], ignore_index=True)
resultados_todos

## 📊 Comparar la coherencia promedio por valor de $k$

In [ ]:
coherencia_promedio = resultados_todos.groupby("k")["coherencia_embeddings"].mean()
coherencia_promedio

In [ ]:
plt.figure(figsize=(6, 4))
coherencia_promedio.plot(kind="bar", color=["#854f0b", "#854f0b", "#854f0b"])
plt.ylabel("Similitud coseno promedio (embeddings)")
plt.xlabel("Número de tópicos (k)")
plt.title("Coherencia semántica promedio por valor de k (embeddings)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## ⚖️ Comparación con el enfoque LLM (5b)

En 5b encontramos que el LLM concentraba casi todos sus puntajes en 5, dejando poco margen para distinguir entre modelos. Vale la pena revisar si este método de embeddings **sí** logra distinguirlos.

Ejecuta esta celda y observa:

- ¿La `coherencia_embeddings` varía más entre tópicos que el puntaje 1-5 del LLM en 5b?
- ¿La tendencia entre $k=12$, $k=20$ y $k=25$ apunta en la misma dirección que en 5b, o difiere?

In [ ]:
resultados_todos.groupby("k")["coherencia_embeddings"].describe()

### 🔍 Lo que muestran los resultados

En una ejecución real, el promedio de `coherencia_embeddings` por valor de $k$ fue:

| $k$ | Coherencia promedio (embeddings) |
|---|---|
| 12 | ≈ 0.265 |
| 20 | ≈ 0.258 |
| 25 | ≈ 0.264 |

La respuesta a las dos preguntas de arriba es, sorprendentemente, **mixta**:

- **Sí, hay más variación entre tópicos individuales.** Los tópicos van de ~0.207 a ~0.338 — un rango real, muy distinto a los puntajes del LLM en 5b, que se concentraban casi todos en 5.
- **Pero no, la tendencia entre valores de $k$ no es más clara — es casi plana.** Las tres medias están a menos de 0.007 entre sí, sin una dirección evidente. En 5b, en cambio, el LLM mostraba una **leve tendencia decreciente** (5.0 → 4.9 → 4.68) a medida que aumentaba $k$, a pesar de tener puntajes mucho más comprimidos dentro de cada modelo.

Esto es un hallazgo interesante en sí mismo: **más granularidad dentro de un modelo no implica necesariamente una señal más clara entre modelos**. El promedio simple de similitud coseno entre las 15 palabras principales parece capturar qué tan "compacto" es un tópico en general, pero no necesariamente cómo cambia esa compacidad a medida que se agregan más tópicos en este rango ($k=12$ a $25$). Una métrica más específica — por ejemplo, ponderada por $\beta$, o comparada contra la similitud promedio de palabras aleatorias del corpus como referencia — probablemente sería necesaria para detectar diferencias reales entre valores de $k$.

> 📌 **Idea clave:** ningún método es automáticamente "mejor" por ser más granular o más matemático. Los embeddings resolvieron el problema de discriminación *dentro* de un modelo, pero no el de comparación *entre* modelos — mientras que el LLM, pese a su escala comprimida, sí ofrecía una tendencia (débil) entre valores de $k$. Elegir el método correcto depende de qué pregunta te estás haciendo.

## ✅ Cierre

Este notebook ofrece una **tercera perspectiva** sobre la calidad de los modelos de tópicos, complementando las dos anteriores:

- **R / Quanteda + topicmodels (5a)**: métricas estadísticas del ajuste del modelo (perplexity, elbow method) — objetivas, pero no dicen nada sobre si un tópico "tiene sentido" para un humano.
- **Python + LLM (5b)**: un LLM lee las palabras y opina sobre su coherencia — captura matices semánticos que una métrica puramente numérica no puede, pero su escala 1-5 resultó poco discriminante.
- **Python + Embeddings (este notebook)**: mide la distancia matemática real entre las palabras — produce un número continuo, más fácil de comparar entre modelos, pero es una medida más "mecánica": no entiende *por qué* un grupo de palabras tiene sentido temático, solo qué tan cerca están en un espacio vectorial entrenado de forma general.

> 📌 **Idea clave:** ningún método es suficiente por sí solo. Las métricas estadísticas, el juicio cualitativo de un LLM y la distancia matemática de los embeddings son **tres fuentes de evidencia complementarias** — y comparar sus conclusiones (¿coinciden en cuál $k$ es mejor?, ¿dónde difieren y por qué?) es en sí mismo un ejercicio de análisis crítico central en este curso.